# PADS Dataset Cleaning for Power BI and Tableau

This notebook explains a simple cleaning workflow for the `pads-parkinsons-disease-smartwatch-dataset` folder.

The goal is to convert mixed-format source files into clean CSV tables that are easier to load into Power BI or Tableau.

## Source Data

The dataset contains four main parts:

- `patients/`: patient demographic and clinical JSON files
- `questionnaire/`: questionnaire JSON files with nested item responses
- `movement/`: assessment JSON files with task and file metadata
- `timeseries/`: raw smartwatch sensor `.txt` files

In [ ]:
from pathlib import Path

base_dir = Path('pads-parkinsons-disease-smartwatch-dataset')
for folder_name in ['patients', 'questionnaire', 'movement', 'timeseries']:
    file_count = len(list((base_dir / folder_name).glob('*')))
    print(f'{folder_name}: {file_count} files')

## Cleaning Strategy

Instead of creating one very large flat table, the cleaning script builds a small set of connected CSV tables:

- `patients.csv`
- `questionnaire_answers.csv`
- `questionnaire_summary.csv`
- `movement_sessions.csv`
- `movement_records.csv`
- `timeseries_summary.csv`
- `data_dictionary.csv`

These tables can be joined by `patient_id`, and the movement/timeseries tables can also be joined by task and file information.

In [ ]:
import subprocess

result = subprocess.run(['python3', 'clean_pads_dataset.py'], capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print(result.stderr)

## Preview the Cleaned CSV Files

In [ ]:
import csv
from pathlib import Path

output_dir = Path('pads_cleaned_csv')

def preview_csv(file_name, limit=5):
    print(f'\nPreview: {file_name}')
    with (output_dir / file_name).open('r', encoding='utf-8') as file:
        reader = csv.DictReader(file)
        for index, row in enumerate(reader, start=1):
            print(row)
            if index >= limit:
                break

for file_name in [
    'patients.csv',
    'questionnaire_summary.csv',
    'movement_sessions.csv',
    'timeseries_summary.csv',
]:
    preview_csv(file_name)

## Suggested BI Model

A good starter model for Power BI or Tableau is:

- Use `patients.csv` as the main dimension table.
- Join `questionnaire_answers.csv` and `questionnaire_summary.csv` on `patient_id`.
- Join `movement_sessions.csv`, `movement_records.csv`, and `timeseries_summary.csv` on `patient_id`.
- Use `task_name` and `device_location` to build movement and wrist-based comparisons.